### 希土類Co合金の磁気転移温度

ベイズ線形回帰を用いて希土類Co合金の磁気転移温度をベイズ線形回帰して回帰係数の分布を調べます。

1. 観測量$T$が線形回帰式で書ける。
2. 観測量$T$には線形回帰式分に加えてノイズ $\delta$ が加わる。
3. $\delta$ はガウス分布に従う。

$$
T = (\vec w,\vec X) + \delta
$$

という仮定を置いています。





In [ ]:
g_data_name = "ZBWZ3"  # ReCo, ZBWZ3
g_sigma = 1


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline


def get_data(data_name):
    """get data from a file.

    Args:
        data_name (str): data name.

    Returns:
        pd.DataFrame: data,
        [str]: a list of explanatory variable names,
        str: target variable name.
    """
    if data_name == "ReCo":
        filename = "../data/TC_ReCo_detail_descriptor.csv"
        df = pd.read_csv(filename)
        # LASSO説明変数
        descriptor_names = ['C_R', 'vol_per_atom', 'f4', 'S4f', 'J4f']
        # all
        #descriptor = ['C_R', 'C_T', 'vol_per_atom', 'f4',       'd5', 'L4f', 'S4f', 'J4f', '(g-1)J4f', '(2-g)J4f']
        target_name = "Tc"
    elif data_name == "ZBWZ3":
        df = pd.read_csv("../data_calculated/ZB_WZ_dE_3var.csv")
        descriptor_names = ['desc1', 'desc2', 'desc3']
        target_name = 'dE'

    return df, descriptor_names, target_name


g_df, g_descriptor_names, g_target_name = get_data(g_data_name)


In [ ]:
from sklearn.preprocessing import StandardScaler


def make_XT(df, descriptor_names, target_name):
    """make X and T from dataframe.
    
    An unit matrix is added to X to make T.

    Args:
        df (pd.DataFrame): data.
        descriptor_names ([str]]): a list of explanatory variables.
        target_name (str): target variable name.

    Returns:
        np.ndarray: X,
        np.ndarray: T.
    """
    X = df[descriptor_names].values
    X = StandardScaler().fit_transform(X)
    # 1 を加える
    N = X.shape[0]
    one = np.ones((N, 1))
    print(one.shape)
    X = np.concatenate([X, one], axis=1)

    T = df[target_name].values
    print(X.shape, T.shape)
    return X, T


g_X, g_T = make_XT(g_df, g_descriptor_names, g_target_name)


規格化の確認のため可視化します。

In [ ]:
plt.plot(g_X, "o-")


コードの都合でｗを作っておきます。


In [ ]:
from sklearn.linear_model import LinearRegression


def fit_linearRegression(X, y):
    """linear regression and evaluate coefficient

    Args:
        X (np.array): descriptor
        y (np.array): target value

    Returns:
        np.array: coefficients of the linear model
    """

    reg = LinearRegression(fit_intercept=False)
    reg.fit(X, y)
    print("coef=", reg.coef_.ravel(), "R2=", reg.score(X, y))
    return reg.coef_.ravel()


g_linear_coef = fit_linearRegression(g_X, g_T)


priorの分布を定義します。

In [ ]:
g_w = g_linear_coef
# w =  np.array([0,0,0,0,0,0])


In [ ]:
from scipy.stats import multivariate_normal


def solve_iteratively(w, N, sigma_, X, T):
    """逐次計算手法を用いる。
    式(6),(7)

    Args:
        w0 (np.array): inital coefficients
        N: the maximum number of iterations
        X (np.array): descriptor
        T (np.array): observed target values
        sigma_ (float): a value of sigma for beta

    Returns:
        list: S
        list: m
    """

    # 初期状態 平均(0,0),stddev = (0.1,0.1)
    m_0 = np.zeros(w.shape[0])
    S_0 = np.identity(w.shape[0]) / 0.2**2

    beta_inv = np.identity(w.shape[0])/sigma_**2

    # save data
    Slist = []
    mlist = []

    bar_m_N = m_0.copy()
    bar_S_N = S_0.copy()
    Slist.append(bar_S_N)
    mlist.append(bar_m_N)

    # fit
    for n in range(N):
        x_N, t_N = X[n], T[n]
        bar_m_N1 = bar_m_N.copy()
        bar_S_N1 = bar_S_N.copy()

        # estimate parameters
        bar_S_N1_inv = np.linalg.inv(bar_S_N1)
        bar_S_N_inv = bar_S_N1_inv + x_N.reshape(-1, 1) * np.dot(beta_inv, x_N)

        bar_S_N = np.linalg.inv(bar_S_N_inv)

        bar_m_N = np.dot(bar_S_N,
                         np.dot(bar_S_N1_inv, bar_m_N1) + np.dot(x_N, np.dot(beta_inv, t_N)))

        Slist.append(bar_S_N)
        mlist.append(bar_m_N)

    return Slist, mlist


g_N = g_X.shape[0]

g_Slist1, g_mlist1 = solve_iteratively(g_w, g_N, g_sigma, g_X, g_T)
print("bar S", g_Slist1[-1])
print("bar m", g_mlist1[-1])


平均値がbar m,共分散行列がbar Sで示されています。

std devに直す

In [ ]:
def make_std(Slist1):
    std = []
    for i in range(Slist1[-1].shape[0]):
        std.append(Slist1[-1][i, i])
    std = np.sqrt(std)
    return std


g_std = make_std(g_Slist1)
g_std


目的変数値と予測値を図示します。

In [ ]:
from numpy.random import multivariate_normal
from sklearn.metrics import r2_score


def draw_N_samples(bar_S_N, bar_m_N, X, Y, npull=500, filename=None):
    """draw N samples from the covariance matrix

    Args:
        bar_S_N (np.array): bar S_N
        bar_m_N (np.array): bar m_N
        X (np.array): descriptor
        T (np.array): true target values
        npull (int, optional): the number of samples to draw. Defaults to 500.

    Returns:
        np.array: prediction
    """
    print(X.shape)
    # w平均値を用いた予測
    Ypredict = np.dot(X, bar_m_N)

    print("R2 score", r2_score(Y, Ypredict))

    ys = np.concatenate([Y, Ypredict])

    # yの範囲
    ylim = (ys.min(), ys.max())

    # T vs  Ypredict
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_xlim(ylim)
    ax.set_ylim(ylim)
    ax.plot(Y, Ypredict, "o")
    ax.plot(ylim, ylim)
    ax.set_xlabel("Y(expr.)")
    ax.set_ylabel("Y(predict)")
    fig.show()

    # T vs Ypredict+-'sigma'
    fig, ax = plt.subplots(figsize=(5, 5))

    # mean
    ax.plot(Y, Ypredict, "o")

    # covariant matrixがわかっているが図示用の乱数を用いる。
    # mean, covariant matrixから分布を出して、randomにwをnpull個引く。
    print("draw ", npull)
    w_rand = multivariate_normal(bar_m_N, bar_S_N, size=npull)
    Ypredict = []
    for i, w_rand1 in enumerate(w_rand):
        Y_rand = np.dot(X, w_rand1)
        Ypredict.append(Y_rand)

    Ypredict = np.array(Ypredict).T
    for y0, y in zip(Y, Ypredict):
        ax.plot(y0*np.ones(y.shape[0]), np.sort(y), ".-", alpha=0.1)

    ax.plot(ylim, ylim)
    ax.set_xlabel("Y(expr.)")
    ax.set_ylabel("Y(predict)")
    ax.set_title("range [{:.2f}:{:.2f}]".format(ylim[0], ylim[1]))
    if filename is not None:
        fig.savefig(filename)
        print("save to", filename)
    fig.show()

    return Ypredict


g_bar_S_N = g_Slist1[-1]
g_bar_m_N = g_mlist1[-1]

import os
os.makedirs("image_executed", exist_ok=True)
g_Ypredict = draw_N_samples(g_bar_S_N, g_bar_m_N, g_X, g_T, 
                            filename="image_executed/{}.png".format(g_data_name))
